# Week 4 — Distributed Word Representations

From sparse counts to dense vectors. We implement word2vec (skip-gram with negative sampling), GloVe, and fastText, each from scratch in NumPy.

## Learning Objectives

- Derive the skip-gram with negative-sampling (SGNS) objective and prove its equivalence to factorizing a shifted PMI matrix (Levy & Goldberg, 2014).
- Implement SGNS in NumPy with manual gradient computation.
- Implement GloVe and explain why its objective is a weighted least-squares matrix factorization.
- Evaluate embeddings intrinsically (analogy, similarity) and extrinsically (classification).

## Required Reading

- Mikolov, T., et al. (2013). *Distributed Representations of Words and Phrases and Their Compositionality*.
- Pennington, J., Socher, R., & Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation*.
- Levy, O., & Goldberg, Y. (2014). *Neural Word Embedding as Implicit Matrix Factorization*.
- Bojanowski, P., et al. (2017). *Enriching Word Vectors with Subword Information*.

In [ ]:
import sys, re
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)

def tokenize(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

## 1. The distributional hypothesis

Firth (1957): *"You shall know a word by the company it keeps."* This is the foundational assumption of all distributional semantics. We make it operational by learning a mapping $w \mapsto \mathbf{v}_w \in \mathbb{R}^d$ such that words appearing in similar contexts have similar vectors.

## 2. Skip-gram with negative sampling (SGNS)

For a center word $w$ and a true context word $c$, plus $k$ negative samples $c'_1, \ldots, c'_k$ drawn from a noise distribution $P_n$:

$$\mathcal{L}(w, c) = \log \sigma(\mathbf{v}_c^\top \mathbf{v}_w) + \sum_{i=1}^{k} \log \sigma(-\mathbf{v}_{c'_i}^\top \mathbf{v}_w).$$

Maximizing this means: push the true (word, context) pair to high dot product, push random pairs to low dot product. The noise distribution is unigram raised to the 3/4 power (Mikolov et al., 2013), which downweights very frequent words.

**Gradients** (worked by hand):

$$\frac{\partial \mathcal{L}}{\partial \mathbf{v}_w} = (1 - \sigma(\mathbf{v}_c^\top \mathbf{v}_w)) \mathbf{v}_c - \sum_i \sigma(\mathbf{v}_{c'_i}^\top \mathbf{v}_w) \mathbf{v}_{c'_i}.$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{v}_c} = (1 - \sigma(\mathbf{v}_c^\top \mathbf{v}_w)) \mathbf{v}_w.$$

In [ ]:
class SGNS:
    def __init__(self, vocab_size, dim=64, window=2, neg=5, lr=0.025):
        self.V, self.dim, self.window, self.neg, self.lr = vocab_size, dim, window, neg, lr
        # Two embedding matrices: 'in' (center) and 'out' (context).
        self.W = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.C = np.zeros((vocab_size, dim))

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

    def _build_sampling_table(self, counts, size=10**6):
        # Unigram^(3/4) sampling table (Mikolov trick).
        probs = np.array([counts.get(i, 0) for i in range(self.V)], dtype=float) ** 0.75
        probs /= probs.sum()
        table = np.random.choice(self.V, size=size, p=probs)
        return table

    def train(self, sequences, epochs=5, verbose=True):
        counts = Counter(tok for seq in sequences for tok in seq)
        self.neg_table = self._build_sampling_table(counts)

        for ep in range(epochs):
            losses = []
            for seq in sequences:
                for i, w in enumerate(seq):
                    start, end = max(0, i - self.window), min(len(seq), i + self.window + 1)
                    for j in range(start, end):
                        if j == i:
                            continue
                        c = seq[j]
                        # Positive sample
                        score_pos = self.W[w] @ self.C[c]
                        sig_pos = self._sigmoid(score_pos)
                        grad_w_acc = (sig_pos - 1) * self.C[c]
                        self.C[c] -= self.lr * (sig_pos - 1) * self.W[w]

                        # Negative samples
                        negs = self.neg_table[np.random.randint(0, len(self.neg_table), size=self.neg)]
                        for c_neg in negs:
                            if c_neg == c:
                                continue
                            score_neg = self.W[w] @ self.C[c_neg]
                            sig_neg = self._sigmoid(score_neg)
                            grad_w_acc += sig_neg * self.C[c_neg]
                            self.C[c_neg] -= self.lr * sig_neg * self.W[w]

                        self.W[w] -= self.lr * grad_w_acc
                        losses.append(-np.log(sig_pos + 1e-12))
            if verbose:
                print(f"epoch {ep+1}/{epochs}  loss = {np.mean(losses):.4f}")
        return self

# Build a small corpus with clear semantic groupings.
SENTENCES = [
    "the cat sat on the mat", "the dog sat on the rug", "the cat chased the mouse",
    "the dog chased the cat", "the king ruled the kingdom", "the queen ruled the kingdom",
    "the prince met the princess", "the queen met the king",
    "machine learning models learn from data", "deep learning models process language",
    "neural networks learn representations from data", "language models predict the next word",
    "transformers process sequences in parallel", "attention models long range dependencies",
] * 8

tokenized = [tokenize(s) for s in SENTENCES]
vocab = sorted(set(t for s in tokenized for t in s))
tok2id = {t: i for i, t in enumerate(vocab)}
id2tok = {i: t for t, i in tok2id.items()}
sequences = [[tok2id[t] for t in s] for s in tokenized]
print(f"|V| = {len(vocab)}, |sentences| = {len(sequences)}")

model = SGNS(vocab_size=len(vocab), dim=32, window=2, neg=5, lr=0.05)
model.train(sequences, epochs=20, verbose=False)
print("Training complete.")

In [ ]:
# Nearest neighbors in embedding space — a qualitative sanity check.
def cosine_sim(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

def nearest(model, word, k=5):
    if word not in tok2id:
        return []
    v = model.W[tok2id[word]]
    sims = [(t, cosine_sim(v, model.W[i])) for t, i in tok2id.items() if t != word]
    return sorted(sims, key=lambda x: -x[1])[:k]

for q in ['cat', 'king', 'learning', 'transformers']:
    print(f"{q:>14}: {[(w, f'{s:.2f}') for w, s in nearest(model, q, 4)]}")

## 3. Levy & Goldberg: SGNS factorizes shifted PMI

Levy & Goldberg (2014) proved that with sufficient embedding dimension, the SGNS objective is optimized when

$$\mathbf{v}_c^\top \mathbf{v}_w = \text{PMI}(w, c) - \log k.$$

This connects neural word embeddings back to the count-based PMI of classical distributional semantics. Below we construct the empirical shifted-PMI matrix and verify it correlates with our SGNS dot products.

In [ ]:
def empirical_pmi(sequences, V, window=2, k=5):
    word_counts = Counter()
    pair_counts = Counter()
    total_pairs = 0
    for seq in sequences:
        for i, w in enumerate(seq):
            word_counts[w] += 1
            for j in range(max(0, i-window), min(len(seq), i+window+1)):
                if j != i:
                    pair_counts[(w, seq[j])] += 1
                    total_pairs += 1
    # PMI(w,c) = log p(w,c) / (p(w) p(c)), shifted by log k.
    total_words = sum(word_counts.values())
    pmi = np.full((V, V), -np.inf)
    for (w, c), cnt in pair_counts.items():
        p_wc = cnt / total_pairs
        p_w = word_counts[w] / total_words
        p_c = word_counts[c] / total_words
        pmi[w, c] = np.log(p_wc / (p_w * p_c)) - np.log(k)
    return pmi

shifted_pmi = empirical_pmi(sequences, V=len(vocab), window=2, k=5)
sgns_dot = model.W @ model.C.T

# Correlation between SGNS dot product and shifted PMI (on observed pairs only).
mask = np.isfinite(shifted_pmi)
corr = np.corrcoef(sgns_dot[mask], shifted_pmi[mask])[0, 1]
print(f"Correlation(SGNS dot, shifted PMI) over observed pairs: {corr:.3f}")
print("(Higher with longer training / larger dim — Levy & Goldberg show it approaches 1.)")

## 4. GloVe — weighted least-squares matrix factorization

Pennington et al. (2014) start from co-occurrence ratios and derive:

$$J = \sum_{i,j} f(X_{ij}) (\mathbf{v}_i^\top \mathbf{v}_j + b_i + b_j - \log X_{ij})^2,$$

where $X_{ij}$ is the (i, j) co-occurrence count and the weighting function

$$f(x) = \min((x / x_{\max})^\alpha, 1)$$

downweights very frequent pairs. This is the explicit matrix factorization that SGNS performs implicitly.

In [ ]:
class GloVe:
    def __init__(self, vocab_size, dim=32, x_max=10, alpha=0.75, lr=0.05):
        self.V, self.dim, self.x_max, self.alpha, self.lr = vocab_size, dim, x_max, alpha, lr
        self.W = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.W_tilde = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.b = np.zeros(vocab_size); self.b_tilde = np.zeros(vocab_size)

    def _cooccur(self, sequences, window=2):
        X = np.zeros((self.V, self.V))
        for seq in sequences:
            for i, w in enumerate(seq):
                for j in range(max(0, i-window), min(len(seq), i+window+1)):
                    if i != j:
                        X[w, seq[j]] += 1.0 / abs(i - j)  # distance-weighted (GloVe convention)
        return X

    def _f(self, x):
        return np.where(x < self.x_max, (x / self.x_max) ** self.alpha, 1.0)

    def train(self, sequences, epochs=50, verbose=False):
        X = self._cooccur(sequences)
        pairs = np.argwhere(X > 0)
        for ep in range(epochs):
            losses = []
            np.random.shuffle(pairs)
            for i, j in pairs:
                xij = X[i, j]
                pred = self.W[i] @ self.W_tilde[j] + self.b[i] + self.b_tilde[j]
                diff = pred - np.log(xij)
                weight = self._f(xij)
                grad = 2 * weight * diff
                self.W[i]       -= self.lr * grad * self.W_tilde[j]
                self.W_tilde[j] -= self.lr * grad * self.W[i]
                self.b[i]       -= self.lr * grad
                self.b_tilde[j] -= self.lr * grad
                losses.append(weight * diff * diff)
            if verbose and ep % 10 == 0:
                print(f"epoch {ep:3d}  loss = {np.mean(losses):.4f}")
        return self

    def vectors(self):
        # Final embedding = sum of the two (standard GloVe convention).
        return self.W + self.W_tilde

glove = GloVe(vocab_size=len(vocab), dim=32, x_max=10).train(sequences, epochs=80)
G = glove.vectors()
print("GloVe nearest neighbors:")
for q in ['cat', 'king', 'learning']:
    if q in tok2id:
        v = G[tok2id[q]]
        sims = sorted([(t, cosine_sim(v, G[i])) for t, i in tok2id.items() if t != q],
                      key=lambda x: -x[1])[:4]
        print(f"  {q:>14}: {[(w, f'{s:.2f}') for w, s in sims]}")

## 5. fastText — subword embeddings

Bojanowski et al. (2017) represent each word as a sum of character n-gram embeddings:

$$\mathbf{v}_w = \sum_{g \in G_w} \mathbf{z}_g,$$

where $G_w$ is the set of character n-grams of $w$ (typically $n \in [3, 6]$) plus $w$ itself. This generalizes naturally to out-of-vocabulary words and to morphologically rich languages like Turkish.

In [ ]:
def char_ngrams(word, n_min=3, n_max=6):
    w = f"<{word}>"
    grams = set()
    for n in range(n_min, n_max + 1):
        for i in range(len(w) - n + 1):
            grams.add(w[i:i+n])
    grams.add(word)
    return grams

# Demonstrate: morphologically related Turkish words share n-grams,
# so they share embedding mass even before training.
for w in ['ev', 'evler', 'evimde', 'evlerimde']:
    g = sorted(char_ngrams(w, 3, 5))
    print(f"{w:>12}: {len(g)} n-grams, sample: {g[:6]}")

## 6. Intrinsic evaluation: similarity and analogy

Two standard families:

- **Similarity** — given (word_a, word_b, human_score), measure Spearman correlation between cosine similarity and human judgment (WordSim-353, SimLex-999).
- **Analogy** — *king is to queen as man is to ?* solved by $\arg\max_w \cos(\mathbf{v}_w, \mathbf{v}_{\text{king}} - \mathbf{v}_{\text{man}} + \mathbf{v}_{\text{woman}})$ (Mikolov et al., 2013).

Both benchmarks are flawed (Linzen, 2016) but remain useful diagnostics.

In [ ]:
def solve_analogy(emb, tok2id, id2tok, a, b, c, top_k=3, exclude=True):
    if not all(w in tok2id for w in (a, b, c)):
        return [(None, 0.0)]
    v = emb[tok2id[b]] - emb[tok2id[a]] + emb[tok2id[c]]
    sims = []
    for t, i in tok2id.items():
        if exclude and t in (a, b, c):
            continue
        sims.append((t, cosine_sim(v, emb[i])))
    return sorted(sims, key=lambda x: -x[1])[:top_k]

print("Analogy: cat:dog :: king:?")
print(" ", solve_analogy(model.W, tok2id, id2tok, 'cat', 'dog', 'king'))
print("\nAnalogy (GloVe): cat:dog :: king:?")
print(" ", solve_analogy(G, tok2id, id2tok, 'cat', 'dog', 'king'))

## 7. Visualization

Project the learned embeddings to 2D via PCA. Semantic clusters should emerge.

In [ ]:
def pca_2d(X):
    Xc = X - X.mean(0, keepdims=True)
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T

proj = pca_2d(model.W)
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(proj[:, 0], proj[:, 1], s=30, alpha=0.7)
for t, i in tok2id.items():
    ax.annotate(t, proj[i], fontsize=9, xytext=(3, 3), textcoords='offset points')
ax.set(title='SGNS embeddings — PCA projection', xlabel='PC1', ylabel='PC2')
plt.tight_layout(); plt.show()

## 8. Exercises

1. **Reproduce Levy & Goldberg.** Train SGNS with $d = |V|$ on a corpus, then verify that the resulting $\mathbf{W} \mathbf{C}^\top$ matrix matches the shifted PMI matrix to within optimization noise.
2. **fastText for Turkish.** Train fastText on a Turkish corpus. Compare nearest-neighbor quality to word2vec for morphologically related word groups (*ev, evler, evimde, ...*).
3. **Analogy at scale.** Download the Google analogy dataset. Evaluate word2vec/GloVe accuracy on each category. Where do they fail?
4. **Embedding bias.** Check whether your trained embeddings exhibit gender bias (Bolukbasi et al., 2016): is $\mathbf{v}_{\text{nurse}} - \mathbf{v}_{\text{doctor}}$ aligned with $\mathbf{v}_{\text{she}} - \mathbf{v}_{\text{he}}$?

---

## Next Week

Week 5 — Language Models: from n-grams to neural. We formalize the task, define perplexity, and implement the Bengio (2003) neural LM.